In [1]:
##############
## INITIATE ##
##############

## Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import pickle

#
###

In [10]:
###########################
## SCRIPT CONFIGURATIONS ##
###########################

## Parameters
MODEL_NAME = 'model_resnet_tcgaoct_raw_FTL_5'
PT_METADATA = '/Volumes/Elements/BDI/projects/GSG/metadata/tcga-cnv/master-sheet-2025-04-18.csv'

## Define paths to embeddings
pti = f'/Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/tcga-oct/{MODEL_NAME}/'
PTO = f'/Volumes/Elements/BDI/projects/GSG/outputs/o3_mil/tcga-oct/{MODEL_NAME}/'
os.makedirs(PTO, exist_ok=True)
# pti = '/Users/user/Documents/projects/GlandSeg/outputs/V2/o2_embeddings/tcga_ffpes/model_resnet_tcga_raw_FTL_0/'
embeddings_path = pti + 'embeddings.npy'
identifiers_path = pti + 'image_identifiers.csv'

## Define paths to images
dataset_path = '/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/tcga-oct/'
# dataset_path = '/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/ICGCC/'
# dataset_path = '/Users/user/Documents/projects/GlandSeg/outputs/V2/o1_extracted_glands/tcga_ffpes_V1'
# dataset_path = '/Users/user/Documents/projects/GlandSeg/outputs/V2_0_0_20250120/o1_extracted_glands/debug'

#
###

In [11]:
######################
## LOAD EMBEDDINGS  ##
######################

## Load the embeddings from the .npy file
embeddings = np.load(embeddings_path)

## Load identifiers
identifiers_df = pd.read_csv(identifiers_path)
image_filenames = identifiers_df['filename'].tolist()

## Check the shape of the loaded embeddings and identifiers
print(f'Loaded embeddings shape: {embeddings.shape}')
print(f'Loaded {len(image_filenames)} image identifiers')

#
###

Loaded embeddings shape: (251532, 512)
Loaded 251532 image identifiers


In [8]:
#######################
## FORMAT IDENTIFIER ##
#######################

## Get unique identifiers
split_marker = '.svs' # For TCGA
# split_marker = '_'
image_filenames_short = [fname.split(split_marker)[1] for fname in image_filenames]
# split_marker = '-01' # For TCGA
image_filenames_short = [fname.split(split_marker)[0] for fname in image_filenames] # For TCGA
# image_filenames_long = [fname.split('.svs')[0].replace('.','-') for fname in image_filenames] # For TCGA
unique_ids = np.unique(image_filenames_short)
num_groups = len(unique_ids)
print(num_groups)
# print(image_filenames_long[0])
numeric_ids = np.zeros(len(image_filenames_short))
k = 0
for unique_id in unique_ids:    
    index = np.argwhere(np.array(image_filenames_short) == unique_id)
    numeric_ids[index] = k
    k += 1

## Checks
print(image_filenames_short[0:10])
print(numeric_ids[0:10])
print(len(numeric_ids))

#
###

253
['TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1', 'TCGA-2A-A8VL-01A_1']
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
251532


In [12]:
###############################
## FORMAT EMBEDDINGS FOR MIL ##
###############################

import numpy as np
import pandas as pd

# Load your metadata CSV (assuming it's called metadata.csv)
df = pd.read_csv(PT_METADATA)  # replace with your actual file path
print(df.shape)

# Ensure slide_id column exists
assert 'sample_id_local' in df.columns, "'sample_id_local' must be a column in your metadata CSV."

# For fast lookup: create a mapping from slide_id to its indices in image_filenames_short
from collections import defaultdict

slideid_to_indices = defaultdict(list)
for idx, name in enumerate(image_filenames_short):
    slideid_to_indices[name].append(idx)

# Assemble per-slide_id embedding subsets
embedding_groups = []
slide_ids_used = []

for slide_id in df['sample_id_local']:
    if slide_id in slideid_to_indices:
        indices = slideid_to_indices[slide_id]
        emb_group = np.array([embeddings[i] for i in indices])
        embedding_groups.append(emb_group)
        slide_ids_used.append(slide_id)
    else:
        # Optional: print or track missing slide_ids
        print(f"missing {slide_id}")
        pass  # or print(f"Slide ID {slide_id} not found in image_filenames_short")

# Convert to NumPy objects and save
np.save(PTO + "embedding_groups.npy", np.array(embedding_groups, dtype=object))  # list of arrays
np.save(PTO + "slide_ids.npy", np.array(slide_ids_used))

# Save embeddings grouped by slide_id as a pickle file
with open(PTO + "embedding_groups.pkl", "wb") as f:
    pickle.dump(embedding_groups, f)

#
###

(253, 53)
